# LIBRARY IMPORTS

In [ ]:
import pandas as pd
import numpy as np
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

#LOAD DATASETS

In [ ]:



df1 = pd.read_excel('/content/AG_NEWS.xlsx')
df2 = pd.read_excel('/content/MSME_DATASET.xlsx')

df1.dropna(how='any', inplace=True)
df2.dropna(how='any', inplace=True)
text_col = 'text'
label_col = 'label'

X1 = df1[text_col].astype(str)
y1 = df1[label_col]

X2 = df2[text_col].astype(str)
y2 = df2[label_col]

display(df1.head(10))
display(df2.head(10))

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,Business
1,Carlyle Looks Toward Commercial Aerospace (Reu...,Business
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,Business
3,Iraq Halts Oil Exports from Main Southern Pipe...,Business
4,"Oil prices soar to all-time record, posing new...",Business
5,"Stocks End Up, But Near Year Lows (Reuters) Re...",Business
6,Money Funds Fell in Latest Week (AP) AP - Asse...,Business
7,Fed minutes show dissent over inflation (USATO...,Business
8,Safety Net (Forbes.com) Forbes.com - After ear...,Business
9,Wall St. Bears Claw Back Into the Black NEW Y...,Business


,text,label
0,"ASSISTANT DIRECTOR, ENFORCEMENT DIRECTORATE .....",02_quality_dispute
1,"State of Maharashtra) Office Notes, Office Mem...",03_no_contract
2,Bangalore District Court M/S Cauvery Distribut...,02_quality_dispute
3,M/s Suri Electricals and Ceramics .....Respond...,01_delayed_payment
4,"Delhi High Court Metal Forgings Pvt. Ltd., New...",03_no_contract
5,M/s Cosmas Research Lab Ltd. ... Respondent(s)...,04_partial_payment
6,West Bengal State Micro Small Enterprises Faci...,01_delayed_payment
7,Lok Sabha Debates Regarding Swavalamban Scheme...,04_partial_payment
8,"1.The Chief Engineer (A/C), TWAD Board, No.30,...",04_partial_payment
9,Piyush Periwal & Ors. ...Respondents Present: ...,04_partial_payment


#TOKENIZATION

In [ ]:
def apply_tokenization(text):
    return word_tokenize(text)

tokens_1 = X1.apply(apply_tokenization)
tokens_2 = X2.apply(apply_tokenization)
print("Word found:Dataset 1 ", len(tokens_1))
print(tokens_1)
print("Word found:Dataset 2 ", len(tokens_2))
print(tokens_2)

Word found:Dataset 1  2000
0       [Wall, St., Bears, Claw, Back, Into, the, Blac...
1       [Carlyle, Looks, Toward, Commercial, Aerospace...
2       [Oil, and, Economy, Cloud, Stocks, ', Outlook,...
3       [Iraq, Halts, Oil, Exports, from, Main, Southe...
4       [Oil, prices, soar, to, all-time, record, ,, p...
                              ...                        
1995    [Google, Interview, Is, Draw, for, Latest, Pla...
1996    [Williamson, Gets, Third, Opinion, on, Elbow, ...
1997    [Iran, Threatens, Israel, on, Nuclear, Reactor...
1998    [Grim, funeral, service, held, for, Burundi, m...
1999    [Borders, Profits, Rise, ,, Raises, Outlook, N...
Name: text, Length: 2000, dtype: object
Word found:Dataset 2  2000
0       [ASSISTANT, DIRECTOR, ,, ENFORCEMENT, DIRECTOR...
1       [State, of, Maharashtra, ), Office, Notes, ,, ...
2       [Bangalore, District, Court, M/S, Cauvery, Dis...
3       [M/s, Suri, Electricals, and, Ceramics, .....,...
4       [Delhi, High, Court, Metal, 

In [ ]:
import pandas as pd
import numpy as np
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

#CASE FOLDING

In [ ]:
def apply_case_folding(tokens):
    return [x.lower() for x in tokens]

tokens_1 = tokens_1.apply(apply_case_folding)
tokens_2 = tokens_2.apply(apply_case_folding)
print("Lower case:Dataset 1 ", len(tokens_1))
print(tokens_1)
print("Lower case:Dataset 2 ", len(tokens_2))
print(tokens_2)

Lower case:Dataset 1  2000
0       [wall, st., bears, claw, back, into, the, blac...
1       [carlyle, looks, toward, commercial, aerospace...
2       [oil, and, economy, cloud, stocks, ', outlook,...
3       [iraq, halts, oil, exports, from, main, southe...
4       [oil, prices, soar, to, all-time, record, ,, p...
                              ...                        
1995    [google, interview, is, draw, for, latest, pla...
1996    [williamson, gets, third, opinion, on, elbow, ...
1997    [iran, threatens, israel, on, nuclear, reactor...
1998    [grim, funeral, service, held, for, burundi, m...
1999    [borders, profits, rise, ,, raises, outlook, n...
Name: text, Length: 2000, dtype: object
Lower case:Dataset 2  2000
0       [assistant, director, ,, enforcement, director...
1       [state, of, maharashtra, ), office, notes, ,, ...
2       [bangalore, district, court, m/s, cauvery, dis...
3       [m/s, suri, electricals, and, ceramics, .....,...
4       [delhi, high, court, metal, 

#SYNONYM SUBSTITUTION

In [ ]:
synonyms = {
    "won't": "will not",
    "can't": "cannot",
    "n't": "not",
    "good": "nice",
    "bad": "poor",
    "big": "large",
    "small": "little",
    "happy": "glad",
    "quick": "fast",
    "begin": "start",
    "prices" : "price"
}

def apply_synonym_substitution(tokens):
    replaced_tokens = []

    for t in tokens:
        if t in synonyms:
            t = synonyms[t]

        replaced_tokens.append(t)

    return replaced_tokens

tokens_1 = tokens_1.apply(apply_synonym_substitution)
tokens_2 = tokens_2.apply(apply_synonym_substitution)
print("Substitute Synonyms: Dataset 1 ", len(tokens_1))
print(tokens_1)
print("Substitute Synonyms: Dataset 2 ", len(tokens_2))
print(tokens_2)

Substitute Synonyms: Dataset 1  2000
0       [wall, st., bears, claw, back, into, the, blac...
1       [carlyle, looks, toward, commercial, aerospace...
2       [oil, and, economy, cloud, stocks, ', outlook,...
3       [iraq, halts, oil, exports, from, main, southe...
4       [oil, price, soar, to, all-time, record, ,, po...
                              ...                        
1995    [google, interview, is, draw, for, latest, pla...
1996    [williamson, gets, third, opinion, on, elbow, ...
1997    [iran, threatens, israel, on, nuclear, reactor...
1998    [grim, funeral, service, held, for, burundi, m...
1999    [borders, profits, rise, ,, raises, outlook, n...
Name: text, Length: 2000, dtype: object
Substitute Synonyms: Dataset 2  2000
0       [assistant, director, ,, enforcement, director...
1       [state, of, maharashtra, ), office, notes, ,, ...
2       [bangalore, district, court, m/s, cauvery, dis...
3       [m/s, suri, electricals, and, ceramics, .....,...
4       [delhi, 

#STEMMING

In [ ]:
stemmer = PorterStemmer()

def apply_stemming_and_join(tokens):
    stemmed_words = [stemmer.stem(word) for word in tokens]
    return " ".join(stemmed_words)

stokens_1 = tokens_1.apply(apply_stemming_and_join)
stokens_2= tokens_2.apply(apply_stemming_and_join)
print("Stems: Dataset 1 ", len(tokens_1))
print(stokens_1)
print("Stems: Dataset 2 ", len(tokens_2))
print(stokens_2)

Stems: Dataset 1  2000
0       wall st. bear claw back into the black ( reute...
1       carlyl look toward commerci aerospac ( reuter ...
2       oil and economi cloud stock ' outlook ( reuter...
3       iraq halt oil export from main southern pipeli...
4       oil price soar to all-tim record , pose new me...
                              ...                        
1995    googl interview is draw for latest playboy iss...
1996    williamson get third opinion on elbow ( ap ) a...
1997    iran threaten israel on nuclear reactor ( ap )...
1998    grim funer servic held for burundi massacr vic...
1999    border profit rise , rais outlook new york ( r...
Name: text, Length: 2000, dtype: object
Stems: Dataset 2  2000
0       assist director , enforc director ..... respon...
1       state of maharashtra ) offic note , offic memo...
2       bangalor district court m/ cauveri distributor...
3       m/ suri electr and ceram ..... respond . coram...
4       delhi high court metal forg pvt . lt

#PUNCATION REMOVAL

In [ ]:
import string

def apply_punctuation_removal(text):
    text_remove_mapping = str.maketrans("", "", string.punctuation)

    clean_text = text.translate(text_remove_mapping)

    return clean_text.split()

remp_1 = stokens_1.apply(apply_punctuation_removal)
remp_2 = stokens_2.apply(apply_punctuation_removal)

print("Punctuation Remove : Dataset 1", len(stokens_1))
print(remp_1.head())
print("\nPunctuation Remove : Dataset 2", len(stokens_2))
print(remp_2.head())

Punctuation Remove : Dataset 1 2000
0    [wall, st, bear, claw, back, into, the, black,...
1    [carlyl, look, toward, commerci, aerospac, reu...
2    [oil, and, economi, cloud, stock, outlook, reu...
3    [iraq, halt, oil, export, from, main, southern...
4    [oil, price, soar, to, alltim, record, pose, n...
Name: text, dtype: object

Punctuation Remove : Dataset 2 2000
0    [assist, director, enforc, director, respond, ...
1    [state, of, maharashtra, offic, note, offic, m...
2    [bangalor, district, court, m, cauveri, distri...
3    [m, suri, electr, and, ceram, respond, coram, ...
4    [delhi, high, court, metal, forg, pvt, ltd, ne...
Name: text, dtype: object


#STOP WORDS REMOVAL

In [ ]:
stop_words = set(stopwords.words("english"))

def apply_stopwords_removal(tokens):
    return [word for word in tokens if word not in stop_words]

X1_clean = remp_1.apply(apply_stopwords_removal)
X2_clean = remp_2.apply(apply_stopwords_removal)
print("Stop words remove: Dataset 1 ", len(X1_clean))
print(X1_clean)
print("Stop words remove: Dataset 2 ", len(X2_clean))
print(X2_clean)

Stop words remove: Dataset 1  2000
0       [wall, st, bear, claw, back, black, reuter, re...
1       [carlyl, look, toward, commerci, aerospac, reu...
2       [oil, economi, cloud, stock, outlook, reuter, ...
3       [iraq, halt, oil, export, main, southern, pipe...
4       [oil, price, soar, alltim, record, pose, new, ...
                              ...                        
1995    [googl, interview, draw, latest, playboy, issu...
1996    [williamson, get, third, opinion, elbow, ap, a...
1997    [iran, threaten, israel, nuclear, reactor, ap,...
1998    [grim, funer, servic, held, burundi, massacr, ...
1999    [border, profit, rise, rais, outlook, new, yor...
Name: text, Length: 2000, dtype: object
Stop words remove: Dataset 2  2000
0       [assist, director, enforc, director, respond, ...
1       [state, maharashtra, offic, note, offic, memor...
2       [bangalor, district, court, cauveri, distribut...
3       [suri, electr, ceram, respond, coram, honbl, m...
4       [delhi, high

#DATASET EXPORT

In [ ]:
x1_export_clean = remp_1.apply(lambda x: " ".join(x) if isinstance(x, list) else x)
x2_export_clean = remp_2.apply(lambda x: " ".join(x) if isinstance(x, list) else x)
y1 = df1[label_col]
y2 = df2[label_col]

clean_dataset_1_agnews = pd.DataFrame({
    "text": x1_export_clean,
    "label": y1
})

clean_dataset_2_legaldispute = pd.DataFrame({
    "text": x2_export_clean,
    "label": y2
})

# Export
clean_dataset_1_agnews.to_excel("clean_dataset_1_agnews.xlsx", index=False)
clean_dataset_2_legaldispute.to_excel("clean_dataset_2_legaldispute.xlsx", index=False)

